## importing libraries


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler , LabelEncoder 
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split 
import numpy as np

In [ ]:
import pandas as pd

sales_data = pd.read_csv('train.csv')
sales_data.drop("Is_Open", axis = 1, inplace = True)
sales_data



In [ ]:
store_info = pd.read_csv('store.csv')
df = pd.merge(store_info, sales_data, on="Store_id", how = 'left')
df


In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(by='Date')
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["WeekOfYear"] = df["Date"].dt.isocalendar().week

median_distance = df['DistanceToRivalStore'].median()
df["DistanceToRivalStore"] = df["DistanceToRivalStore"].fillna(median_distance)

df.drop(columns=["Date" , "NumberOfCustomers","Store_id" ] , inplace=True)

for i in range(1, 11):
    df[f"Sales {i} day ago"] = df['Sales'].shift(i)
# adding the previous 10 days moving avarage
df["Sales_Mean10"] = df["Sales"].rolling(10).mean() 
df.dropna(inplace=True)

Y = df["Sales"]
X = df.drop(columns=["Sales"])




In [ ]:
numeric_features = X[["DistanceToRivalStore", "RivalOpeningMonth",  "ContinuousBogo", "ContinuousBogoSinceWeek","BOGO", "Holiday", "Year", "Month", "Day", "WeekOfYear", "RivalEntryYear", "ContinuousBogoSinceYear","Sales_Mean10"] + [f"Sales {i} day ago" for i in range(1, 11)]]

scaler = StandardScaler()
scaler.fit_transform(numeric_features)

le = LabelEncoder()

X["RetailType"] = le.fit_transform(X["RetailType"].astype(str))
X["Stock variety"] = le.fit_transform(X["Stock variety"].astype(str))
X["ContinuousBogoMonths"] = le.fit_transform(X["ContinuousBogoMonths"].astype(str))


X_train , X_test , Y_train , Y_test = train_test_split(X, Y , shuffle=False , random_state=None , train_size=0.7)


In [ ]:
tree = RandomForestRegressor()

tree.fit(X_train, Y_train)
y_pred_rf = tree.predict(X_test)


linear_model = LinearRegression()

param_space = {
    'fit_intercept': [True, False],
    'positive': [True, False],
}
grid_search =  GridSearchCV(estimator=linear_model, param_grid = param_space, cv = 2, n_jobs=-2)


grid_search.fit(X_train, Y_train)  
y_pred_linear = grid_search.predict(X_test)

In [ ]:
# rmspe
print(f"RMSPE linear regression : {np.sqrt(np.mean(((Y_test - y_pred_linear)/(Y_test + 1))**2)) * 100}")
print("linear_model score", grid_search.best_score_)

In [ ]:
feature_importances = tree.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importances
})

plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importance from Random Forest Model')
plt.show()

print("Most important features:")
print(feature_importance_df)